# 03 — Evaluate DDPM and Flow Matching

Assumes you've already trained both models (notebook 02).

Produces all figures and tables for the report:
1. Qualitative samples (side-by-side, real vs DDPM vs FM)
2. Radial energy spectrum comparison
3. Vorticity PDF comparison
4. Sampling efficiency sweep (NFE vs quality)

In [ ]:
# --- Bootstrap (same as other notebooks) ---
import sys
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    REPO = Path('/content/drive/MyDrive/courses/24788-Intro_of_DL/project/code')
except ImportError:
    IN_COLAB = False
    REPO = Path.cwd()
    while not (REPO / 'requirements.txt').exists() and REPO != REPO.parent:
        REPO = REPO.parent

sys.path.insert(0, str(REPO))
print(f'IN_COLAB = {IN_COLAB}\nREPO     = {REPO}')

In [ ]:
import json, numpy as np, torch, matplotlib.pyplot as plt, h5py
from src.env import CKPT_DIR, RESULTS_DIR, DATA_DIR
from src.data import KolmFlowFrames, NormStats, get_trajectory_split, DEFAULT_PATH
from src.unet import UNet
from src.diffusion import DDPM
from src.flow_matching import FlowMatching
from src.train import TrainConfig, build_model
from src import evaluate as E

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device = {device}')

with open(DATA_DIR / 'norm_stats.json') as f:
    d = json.load(f); norm = NormStats(mean=d['mean'], std=d['std'])
print(f'norm: mean={norm.mean:.4f} std={norm.std:.4f}')

In [ ]:
# --- Load both trained models (EMA weights) ---
def load_model(model_type, run_name='main'):
    run_dir = CKPT_DIR / f'{model_type}_{run_name}'
    # Prefer the standalone EMA snapshot (what graders download from Box);
    # fall back to state.pt if a full training state is present locally.
    ema_path = run_dir / 'ema_step10000.pt'
    state_path = run_dir / 'state.pt'
    src_path = ema_path if ema_path.exists() else state_path
    state = torch.load(src_path, map_location=device, weights_only=False)
    cfg_dict = state['config']
    cfg = TrainConfig(**{k: v for k, v in cfg_dict.items() if k in TrainConfig.__annotations__})
    cfg.ch_mults = tuple(cfg.ch_mults)
    wrapper, net = build_model(cfg)
    ema_sd = state['ema']
    for n, p in net.named_parameters():
        if n in ema_sd:
            p.data.copy_(ema_sd[n].to(device))
    wrapper.to(device).eval()
    print(f'Loaded {model_type} from {src_path.name}, step {state["step"]}')
    return wrapper, cfg

ddpm, cfg_ddpm = load_model('ddpm')
fm,   cfg_fm   = load_model('fm')

In [ ]:
# --- Draw a batch of real samples for comparison ---
N_EVAL = 256  # enough to stabilize distributional metrics
train_ids, val_ids, test_ids = get_trajectory_split()

real_frames = []
with h5py.File(DEFAULT_PATH, 'r') as h5:
    u = h5['valid']['u']
    rng = np.random.default_rng(42)
    for i in test_ids:
        for t in rng.choice(200, size=N_EVAL // len(test_ids) + 1, replace=False):
            real_frames.append(np.asarray(u[int(i), int(t)], dtype=np.float32))
real_frames = np.stack(real_frames[:N_EVAL])
print(f'real_frames: {real_frames.shape}')
real_norm = (real_frames - norm.mean) / norm.std

In [ ]:
# --- Generate samples from each model (match default NFE) ---
DDPM_NFE = 1000  # full ancestral sampling
FM_NFE   = 50    # Euler, already competitive

with torch.no_grad():
    ddpm_samples, _ = E.generate_samples(ddpm, N_EVAL, 160, DDPM_NFE, device, 'ddpm')
    fm_samples,   _ = E.generate_samples(fm,   N_EVAL, 160, FM_NFE,   device, 'fm')

# Denormalize for physical-space metrics
ddpm_denorm = ddpm_samples * norm.std + norm.mean
fm_denorm   = fm_samples   * norm.std + norm.mean
print(f'DDPM samples: {ddpm_samples.shape}  std(denorm)={ddpm_denorm.std():.3f}')
print(f'FM   samples: {fm_samples.shape}  std(denorm)={fm_denorm.std():.3f}')
print(f'real            : std(denorm)={real_frames.std():.3f}')

In [ ]:
# --- Qualitative panel: real / DDPM / FM, 6 samples each ---
# Per-image colormap so DDPM's extreme outliers don't squash real/FM
# rows to white. The annotated |max| makes the magnitude gap visible.
n_show = 6
fig, axes = plt.subplots(3, n_show, figsize=(2 * n_show, 6.8))
rows = [
    ('real',          real_frames),
    ('DDPM',          ddpm_denorm),
    ('Flow Matching', fm_denorm),
]
for r, (label, arr) in enumerate(rows):
    for j in range(n_show):
        v = float(np.abs(arr[j]).max())
        axes[r, j].imshow(arr[j], cmap='RdBu_r', vmin=-v, vmax=v)
        axes[r, j].axis('off')
        txt = f'|max|={v:.2g}' if v < 100 else f'|max|={v:.1e}'
        axes[r, j].text(
            0.02, 0.96, txt, transform=axes[r, j].transAxes,
            fontsize=8, va='top', ha='left',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1.5),
        )
    axes[r, 0].set_title(label, loc='left', fontsize=11)
fig.suptitle('Generated vorticity fields (per-image colormap; |max| annotated)', y=1.02)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'samples_panel.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# --- Energy spectrum comparison ---
k, E_real = E.radial_energy_spectrum(real_frames)
_, E_ddpm = E.radial_energy_spectrum(ddpm_denorm)
_, E_fm   = E.radial_energy_spectrum(fm_denorm)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(k, E_real, 'k-',  lw=2,   label='real')
ax.loglog(k, E_ddpm, 'C0--', lw=1.5, label=f'DDPM (NFE={DDPM_NFE})')
ax.loglog(k, E_fm,   'C1:',  lw=1.5, label=f'Flow Matching (NFE={FM_NFE})')
ax.set_xlabel('wavenumber k'); ax.set_ylabel('E(k)')
ax.set_title('Radial energy spectrum'); ax.legend(); ax.grid(alpha=0.3, which='both')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'energy_spectrum.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'log-spectrum L1  DDPM: {E.log_spectrum_error(E_real, E_ddpm):.4f}')
print(f'log-spectrum L1  FM  : {E.log_spectrum_error(E_real, E_fm):.4f}')

In [ ]:
# --- Vorticity PDF comparison ---
vmax = max(abs(real_frames).max(), abs(ddpm_denorm).max(), abs(fm_denorm).max())
edges, h_real = E.vorticity_histogram(real_frames, bins=120, value_range=(-vmax, vmax))
_,     h_ddpm = E.vorticity_histogram(ddpm_denorm, bins=120, value_range=(-vmax, vmax))
_,     h_fm   = E.vorticity_histogram(fm_denorm,   bins=120, value_range=(-vmax, vmax))
centers = 0.5 * (edges[:-1] + edges[1:])

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(centers, h_real, 'k-',  lw=2,   label='real')
ax.plot(centers, h_ddpm, 'C0--', lw=1.5, label='DDPM')
ax.plot(centers, h_fm,   'C1:',  lw=1.5, label='Flow Matching')
ax.set_yscale('log'); ax.set_xlabel('vorticity ω'); ax.set_ylabel('density')
ax.set_title('Marginal vorticity distribution'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'vorticity_pdf.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Wasserstein-1  DDPM vs real: {E.wasserstein1(ddpm_denorm, real_frames):.4f}')
print(f'Wasserstein-1  FM   vs real: {E.wasserstein1(fm_denorm,   real_frames):.4f}')

In [ ]:
# --- Sampling efficiency sweep: quality vs NFE ---
NFE_VALUES = [5, 10, 20, 50, 100, 200, 500]
N_SWEEP = 128  # smaller for speed

real_small = real_frames[:N_SWEEP]

results = []
for nfe in NFE_VALUES:
    for mt, wrapper in [('ddpm', ddpm), ('fm', fm)]:
        gen, eff_nfe = E.generate_samples(wrapper, N_SWEEP, 160, nfe, device, mt)
        gen_denorm = gen * norm.std + norm.mean
        res = E.evaluate_against_real(gen_denorm, real_small)
        results.append({
            'model': mt, 'nfe_requested': nfe, 'nfe_effective': eff_nfe,
            'log_spec_l1': res.log_spectrum_l1,
            'wasserstein': res.wasserstein_vorticity,
            'gen_std': res.gen_std, 'real_std': res.real_std,
        })
        print(f'[{mt:4s}] NFE={nfe:4d}  spec_L1={res.log_spectrum_l1:.3f}  W1={res.wasserstein_vorticity:.3f}')

import pandas as pd
df = pd.DataFrame(results)
df.to_csv(RESULTS_DIR / 'nfe_sweep.csv', index=False)
df

In [ ]:
# --- Plot NFE-vs-quality curves ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for mt, color, marker in [('ddpm', 'C0', 'o'), ('fm', 'C1', 's')]:
    sub = df[df['model'] == mt]
    axes[0].semilogx(sub['nfe_effective'], sub['log_spec_l1'], marker=marker, color=color, label=mt.upper())
    axes[1].semilogx(sub['nfe_effective'], sub['wasserstein'], marker=marker, color=color, label=mt.upper())
axes[0].set_xlabel('NFE'); axes[0].set_ylabel('log-spectrum L1'); axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].set_xlabel('NFE'); axes[1].set_ylabel('Wasserstein-1 (vorticity)'); axes[1].grid(alpha=0.3); axes[1].legend()
fig.suptitle('Sampling efficiency: quality vs number of function evaluations')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'nfe_sweep.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# --- Save summary JSON for the report ---
summary = {
    'n_eval_samples': N_EVAL,
    'ddpm_default_nfe': DDPM_NFE,
    'fm_default_nfe':   FM_NFE,
    'log_spec_l1': {
        'ddpm': E.log_spectrum_error(E_real, E_ddpm),
        'fm':   E.log_spectrum_error(E_real, E_fm),
    },
    'wasserstein1_vorticity': {
        'ddpm': E.wasserstein1(ddpm_denorm, real_frames),
        'fm':   E.wasserstein1(fm_denorm,   real_frames),
    },
    'real_std': float(real_frames.std()),
    'ddpm_std': float(ddpm_denorm.std()),
    'fm_std':   float(fm_denorm.std()),
}
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
summary